In [ ]:
!nvidia-smi

In [ ]:
import ctypes, os, time
from pathlib import Path

# 1. Load DuckDB with Sirius
EXT = Path.home() / "sirius/build/release/extension/sirius/sirius.duckdb_extension"
os.environ["SIRIUS_PROTOTYPE_BACKEND"] = str(EXT)


In [ ]:
import _duckdb, duckdb
ctypes.CDLL(_duckdb.__file__, mode=ctypes.RTLD_GLOBAL)

con = duckdb.connect("trips.duckdb", config={"allow_unsigned_extensions": "true",
                                            "allow_extensions_metadata_mismatch": "true"})
con.execute(f"LOAD '{EXT}'")

In [ ]:
# 2. Make a bigger table: 20 copies of one month = ~60 million rows
#    (only created the first time; saved in trips.duckdb after that)
con.execute("""
    CREATE TABLE IF NOT EXISTS trips AS
    SELECT passenger_count, trip_distance, payment_type,
           tip_amount, PULocationID, total_amount
    FROM './data/yellow_tripdata_2024-01.parquet', range(20)
""")
print("Rows:", con.sql("SELECT count(*) FROM trips").fetchone()[0])

In [ ]:
# 3. Save to disk, then keep the table in GPU memory
con.execute("CHECKPOINT")
con.execute("CALL pin_table(format='duckdb', name='trips', tier='gpu')")

In [ ]:
# 4. Queries to test
queries = {
    "Filter + average": """
        SELECT payment_type, AVG(tip_amount)
        FROM trips WHERE trip_distance > 5
        GROUP BY payment_type""",

    "Many groups": """
        SELECT PULocationID, passenger_count, payment_type, ROUND(trip_distance) AS miles,
               COUNT(*) AS n, AVG(total_amount) AS avg_total
        FROM trips
        GROUP BY PULocationID, passenger_count, payment_type, miles
        ORDER BY n DESC LIMIT 10""",

    "Join": """
        SELECT t.PULocationID, COUNT(*) AS above_avg_trips
        FROM trips t
        JOIN (SELECT PULocationID, AVG(total_amount) AS avg_total
              FROM trips GROUP BY PULocationID) z
          ON t.PULocationID = z.PULocationID
        WHERE t.total_amount > z.avg_total
        GROUP BY t.PULocationID
        ORDER BY above_avg_trips DESC LIMIT 10""",

    "Heavy math": """
        SELECT payment_type,
               SUM(tip_amount * tip_amount)           AS a,
               AVG(trip_distance * total_amount)      AS b,
               SUM(tip_amount / (total_amount + 1.0)) AS c
        FROM trips WHERE trip_distance BETWEEN 1 AND 50
        GROUP BY payment_type""",
}

In [ ]:
# 5. Time a query (warm up once, then time it)
def run(sql):
    con.execute(sql).fetchall()
    start = time.time()
    con.execute(sql).fetchall()
    return time.time() - start


In [ ]:
# 6. Compare CPU and GPU
for name, sql in queries.items():
    con.execute("SET gpu_execution = false")
    cpu = run(sql)
    con.execute("SET gpu_execution = true")
    gpu = run(sql)
    print(f"{name}: CPU {cpu:.3f}s | GPU {gpu:.3f}s | {cpu/gpu:.1f}x faster")